# Klasifikasi Penggunaan dan Tutupan Lahan (LULC) Jawa Timur
## Berbasis Sampel Piksel Sentinel-2 L2A (Copernicus Data Space) — Gaussian Naive Bayes vs K-Nearest Neighbors

**UTS Sistem Informasi Geografis:** analisis spasial penggunaan lahan dan kebijakan.

Alur notebook: **Data Understanding → Pembacaan poligon → Sampel piksel → Ekstraksi Sentinel-2 (Copernicus) → Validasi data → Split 70/15/15 → Model 1 (Naive Bayes) → Model 2 (KNN) → Evaluasi → Peta per model → Luas per kelas.**

---
## Data Understanding

### Tujuan analisis
Memetakan penggunaan dan tutupan lahan (*land use / land cover*, LULC) Provinsi Jawa Timur ke dalam 6 kelas secara otomatis dari citra satelit Sentinel-2, sebagai dasar analisis spasial untuk kebijakan tata ruang, misalnya pemantauan alih fungsi sawah, perlindungan mangrove, dan pengawasan perairan darat (ranu/danau).

### Kolekting data
| Komponen | Keterangan |
|---|---|
| Citra | Sentinel-2 Level-2A (reflektansi permukaan / BOA) dari **Copernicus Data Space Ecosystem**, diakses lewat **Sentinel Hub Process API** |
| Komposit | Median temporal periode `TGL_MULAI`-`TGL_AKHIR` (musim kemarau 2025), scene dengan tutupan awan > `MAX_CLOUD`% dibuang |
| Masking | Piksel awan, bayangan awan, dan cirrus dibuang memakai band `SCL` (kelas 3, 8, 9, 10) |
| Label | Poligon hasil digitasi 6 kelas (shapefile), dipotong ke area studi |
| Unit observasi | **Satu piksel Sentinel-2 (10 m)** pada titik acak di dalam poligon |
| Pembagian data | 70% training, 15% validasi, 15% test, dibagi **per poligon** agar tidak terjadi kebocoran spasial |

### Kelas dan acuan klasifikasi di Indonesia
Kelas penelitian dipetakan ke acuan klasifikasi penutup lahan nasional (**SNI 7645-1:2014** dan klasifikasi penutupan lahan KLHK):

| Kelas penelitian | Padanan pada acuan nasional (berdasarkan nama) |
|---|---|
| Sawah | Daerah pertanian — sawah |
| Bangunan | Permukiman dan lahan bukan pertanian yang berkaitan (lahan terbangun) |
| Mangrove | Hutan mangrove |
| Lahan Hijau | Vegetasi non-sawah: hutan lahan kering, hutan tanaman, perkebunan, semak/belukar |
| Perairan Terbuka (Laut) | Perairan — laut |
| Danau | Perairan — danau, waduk, ranu |

> Catatan: padanan di atas disusun berdasarkan nama kelas. Periksa kembali istilah dan kode persisnya pada dokumen SNI 7645-1:2014 / KLHK sebelum dikumpulkan.

---
## Deskripsi Fitur Sentinel-2A dan Rumus Indeks Spektral
Dalam penginderaan jauh multispektral, fitur yang diekstraksi terdiri atas **10 Band Reflektansi Permukaan (Bottom-Of-Atmosphere / BOA L2A)** dan **8 Indeks Spektral Turunan**:

### Band Reflektansi Sentinel-2A
| Band | Panjang Gelombang | Resolusi | Karakteristik Utama |
|---|---|---|---|
| `B2` (Blue) | 492 nm | 10 m | Sensitif terhadap hamburan atmosfer dan penetrasi air jernih |
| `B3` (Green) | 560 nm | 10 m | Puncak pantulan klorofil vegetasi & badan air |
| `B4` (Red) | 665 nm | 10 m | Penyerapan klorofil tanaman (kunci NDVI) |
| `B5` (Red Edge 1) | 704 nm | 20 m | Batas serapan merah - pantulan inframerah |
| `B6` (Red Edge 2) | 740 nm | 20 m | Peningkatan reflektansi struktur sel daun |
| `B7` (Red Edge 3) | 783 nm | 20 m | Plateau inframerah dekat |
| `B8` (NIR) | 833 nm | 10 m | Pantulan tertinggi biomassa sel daun |
| `B8A` (Narrow NIR) | 865 nm | 20 m | Deteksi uap air dan kanopi sempit |
| `B11` (SWIR 1) | 1614 nm | 20 m | Sensitif terhadap kelembaban tanaman & lahan terbangun |
| `B12` (SWIR 2) | 2202 nm | 20 m | Deteksi mineral, batuan, dan tanah terbuka |

### Rumus 8 Indeks Spektral
| Fitur | Nama Lengkap | Rumus Matematis | Fungsi Analisis |
|---|---|---|---|
| **`NDVI`** | Normalized Difference Vegetation Index | $\dfrac{B8 - B4}{B8 + B4}$ | Mengukur kehijauan & kerapatan kanopi vegetasi |
| **`NDWI`** | Normalized Difference Water Index (McFeeters) | $\dfrac{B3 - B8}{B3 + B8}$ | Membedakan badan air dengan daratan |
| **`MNDWI`**| Modified NDWI (Xu) | $\dfrac{B3 - B11}{B3 + B11}$ | Membedakan badan air dari lahan terbangun/pemukiman |
| **`NDBI`** | Normalized Difference Built-up Index | $\dfrac{B11 - B8}{B11 + B8}$ | Deteksi konsentrasi bangunan & semen/aspal |
| **`NDRE`** | Normalized Difference Red Edge | $\dfrac{B8 - B5}{B8 + B5}$ | Deteksi klorofil tanaman & kanopi lebat |
| **`EVI`**  | Enhanced Vegetation Index | $2.5 \times \dfrac{B8 - B4}{B8 + 6 \cdot B4 - 7.5 \cdot B2 + 1}$ | Vegetasi lebat dengan koreksi gangguan atmosfer |
| **`SAVI`** | Soil Adjusted Vegetation Index | $1.5 \times \dfrac{B8 - B4}{B8 + B4 + 0.5}$ | Vegetasi dengan koreksi latar belakang tanah ($L=0.5$) |
| **`BSI`**  | Bare Soil Index | $\dfrac{(B11 + B4) - (B8 + B2)}{(B11 + B4) + (B8 + B2)}$ | Identifikasi tanah kosong, pasir, dan lahan kering |

---
## Import Pustaka, Konfigurasi, dan Fungsi Indeks Spektral

In [ ]:
%pip install -q -U sentinelhub geopandas shapely pyproj rasterio folium scikit-learn seaborn matplotlib

In [ ]:
import os, io, glob, time, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# ===== KONFIGURASI UMUM =====
DATA_DIR = "."
SEED = 42
# Area studi (kotak Jawa Timur + perairan sekitarnya). Poligon dipotong ke kotak ini.
AOI = dict(west=110.7, south=-8.8, east=114.45, north=-6.6)
# ============================

KELAS = {
    1: {"nama": "Sawah",                   "folder": "sawah",      "file": "sawah",      "warna": "#FFD92F"},
    2: {"nama": "Bangunan",                "folder": "bangunan",   "file": "bangunan",   "warna": "#E41A1C"},
    3: {"nama": "Mangrove",                "folder": "mangrove",   "file": "mangrove",   "warna": "#8E44AD"},
    4: {"nama": "Lahan Hijau",             "folder": "lahanhijau", "file": "lahanhijau", "warna": "#2E7D32"},
    5: {"nama": "Perairan Terbuka (Laut)", "folder": "laut",       "file": "laut",       "warna": "#0D47A1"},
    6: {"nama": "Danau",                   "folder": "danau",      "file": "danau",      "warna": "#4FC3F7"},
}
NAMA_KELAS  = [v["nama"] for v in KELAS.values()]
WARNA_KELAS = {v["nama"]: v["warna"] for v in KELAS.values()}
NAMA_KE_ID  = {v["nama"]: k for k, v in KELAS.items()}

BAND_ASLI = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
INDEKS_SPEKTRAL = ["NDVI", "NDWI", "MNDWI", "NDBI", "NDRE", "EVI", "SAVI", "BSI"]
FITUR_SEMUA = BAND_ASLI + INDEKS_SPEKTRAL

def hitung_indeks_spektral(df_input):
    # Band harus berupa reflektansi skala 0-1
    d = df_input.copy()
    eps = 1e-9
    nd = lambda a, b: (a - b) / (a + b + eps)
    d["NDVI"]  = nd(d["B8"], d["B4"])
    d["NDWI"]  = nd(d["B3"], d["B8"])
    d["MNDWI"] = nd(d["B3"], d["B11"])
    d["NDBI"]  = nd(d["B11"], d["B8"])
    d["NDRE"]  = nd(d["B8"], d["B5"])
    d["EVI"]   = 2.5 * (d["B8"] - d["B4"]) / (d["B8"] + 6.0 * d["B4"] - 7.5 * d["B2"] + 1.0 + eps)
    d["SAVI"]  = 1.5 * (d["B8"] - d["B4"]) / (d["B8"] + d["B4"] + 0.5 + eps)
    d["BSI"]   = (((d["B11"] + d["B4"]) - (d["B8"] + d["B2"])) /
                  ((d["B11"] + d["B4"]) + (d["B8"] + d["B2"]) + eps))
    return d

print("Total kelas :", len(KELAS))
print("Total fitur :", len(FITUR_SEMUA), "(10 band + 8 indeks)")

---
## Pembacaan dan Validasi Poligon
Shapefile 6 kelas dibaca, CRS dinormalkan ke EPSG:4326, geometri invalid diperbaiki, lalu **dipotong ke area studi (AOI)**. Poligon yang berada di luar AOI dibuang dan dilaporkan (kemungkinan salah digitasi).

In [ ]:
from shapely.geometry import box
from shapely.validation import make_valid

def cari_shp(nama):
    # cari file .shp di DATA_DIR dan subfoldernya (tidak peka huruf besar/kecil)
    target = nama.lower().replace(" ", "").replace("_", "")
    for root, _, files in os.walk(DATA_DIR):
        for f in files:
            n = f[:-4].lower().replace(" ", "").replace("_", "")
            if f.lower().endswith(".shp") and (n == target or target in n or n in target):
                return os.path.join(root, f)
    raise FileNotFoundError(f"File {nama}.shp tidak ditemukan di '{os.path.abspath(DATA_DIR)}'")

def baca_shp(path):
    g = gpd.read_file(path)
    g = g.set_crs(4326) if g.crs is None else g.to_crs(4326)
    g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g["geometry"] = g.geometry.apply(lambda x: x if x.is_valid else make_valid(x))
    return g[["geometry"]].reset_index(drop=True)

AOI_BOX = box(AOI["west"], AOI["south"], AOI["east"], AOI["north"])

parts, rekap = [], []
for k, v in KELAS.items():
    g = baca_shp(cari_shp(v["file"]))
    n_awal = len(g)
    g = gpd.clip(g, AOI_BOX)                                   # potong ke area studi
    g = g.explode(index_parts=False).reset_index(drop=True)
    g = g[g.geom_type == "Polygon"].reset_index(drop=True)
    g["kelas_id"], g["kelas"] = k, v["nama"]
    parts.append(g)
    rekap.append({"ID": k, "Kelas": v["nama"], "Poligon awal": n_awal, "Poligon di dalam AOI": len(g)})

gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs=4326)
gdf["poly_id"] = range(1, len(gdf) + 1)

utm = gdf.to_crs(32749)                                        # UTM 49S (meter)
gdf["area_ha"]       = (utm.area / 10000).round(4)
gdf["est_pixel_10m"] = (utm.area / 100).round(0).astype("int64")
rp = gdf.geometry.representative_point()
gdf["lon"], gdf["lat"] = rp.x.round(6), rp.y.round(6)

df_rekap = pd.DataFrame(rekap).set_index("ID")
df_rekap["Total Luas (Ha)"] = gdf.groupby("kelas_id").area_ha.sum().round(1)
display(df_rekap)
print("Total poligon dalam AOI:", len(gdf))
hilang = (df_rekap["Poligon awal"] - df_rekap["Poligon di dalam AOI"]).clip(lower=0).sum()
if hilang > 0:
    print(f"[!] {hilang} poligon berada di luar AOI dan dibuang. Cek apakah itu salah digitasi.")

---
## Pengambilan Sampel Piksel
Titik acak dibuat di dalam poligon dengan jatah sama rata per poligon, agar poligon besar (misalnya laut) tidak mendominasi. Titik yang jatuh di piksel 10 m yang sama dibuang. **Satu titik = satu piksel Sentinel-2.** `N_PER_CLASS` diubah-ubah untuk eksperimen jumlah data (misalnya 500, 1500, 3000).

In [ ]:
from shapely.geometry import Point
from shapely.prepared import prep
from pyproj import Transformer

# ===== KONFIGURASI EKSPERIMEN JUMLAH DATA =====
N_PER_CLASS  = 1500    # target titik (piksel) per kelas -> coba 500 / 1500 / 3000
MAX_PER_POLY = 150     # batas titik per poligon
# ==============================================

rng = np.random.default_rng(SEED)

def rand_pts(poly, n, max_try=200):
    minx, miny, maxx, maxy = poly.bounds
    pp = prep(poly)
    out, tries = [], 0
    while len(out) < n and tries < n * max_try:
        tries += 1
        p = Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy))
        if pp.contains(p):
            out.append(p)
    return out

rows = []
for kid, grp in gdf.groupby("kelas_id"):
    n_tiap = int(np.ceil(N_PER_CLASS / len(grp)))              # jatah sama rata per poligon
    for _, r in grp.iterrows():
        n = int(min(n_tiap, MAX_PER_POLY, max(1, r.est_pixel_10m)))
        for p in rand_pts(r.geometry, n):
            rows.append({"poly_id": r.poly_id, "kelas_id": r.kelas_id, "kelas": r.kelas,
                         "lon": p.x, "lat": p.y})

pts = pd.DataFrame(rows)

# buang titik yang jatuh di piksel 10 m yang sama
to_utm = Transformer.from_crs(4326, 32749, always_xy=True)
x, y = to_utm.transform(pts.lon.values, pts.lat.values)
pts["gx"], pts["gy"] = np.floor(x / 10).astype(int), np.floor(y / 10).astype(int)
pts = (pts.drop_duplicates(["gx", "gy"]).drop(columns=["gx", "gy"])
          .sort_values(["kelas_id", "poly_id"]).reset_index(drop=True))

print("Total titik sampel:", len(pts))
display(pts.groupby("kelas").agg(titik=("poly_id", "size"), poligon=("poly_id", "nunique")).reindex(NAMA_KELAS))

---
## Ekstraksi Nilai Sentinel-2 dari Copernicus Data Space Ecosystem

**Persiapan (sekali saja):**
1. Daftar akun di https://identity.dataspace.copernicus.eu (tombol *register*).
2. Buka **Sentinel Hub Dashboard**: https://shapps.dataspace.copernicus.eu/dashboard/ → **User settings** → **OAuth clients** → buat client baru. Salin **Client ID** dan **Client Secret** (secret hanya tampil sekali).
3. Jalankan sel login di bawah dan tempel keduanya saat diminta.

**Cara kerja:** titik dikelompokkan per poligon menjadi potongan citra kecil (*chip*). Tiap chip diminta ke Copernicus sebagai komposit median Sentinel-2, lalu nilai piksel diambil di titik-titik di dalamnya. Dengan cara ini tidak ada file TIF besar yang diunduh dan jumlah request tetap hemat.

**Biaya:** akun gratis punya kuota bulanan (*Processing Units* dan jumlah request). Sel estimasi menghitung perkiraan biayanya sebelum data ditarik. Pantau pemakaian aktual di dashboard.

In [ ]:
from sentinelhub import SHConfig
import getpass

# Isi sekali saja. Kredensial tersimpan di profil "cdse" pada komputermu (~/.config/sentinelhub/config.toml)
try:
    config = SHConfig("cdse")
    assert config.sh_client_id and config.sh_client_secret
    print("Profil 'cdse' ditemukan, memakai kredensial tersimpan.")
except Exception:
    config = SHConfig()
    config.sh_client_id     = getpass.getpass("OAuth client ID     : ")
    config.sh_client_secret = getpass.getpass("OAuth client secret : ")
    config.sh_base_url  = "https://sh.dataspace.copernicus.eu"
    config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    config.save("cdse")
    print("Kredensial disimpan ke profil 'cdse'.")

print("Base URL:", config.sh_base_url)

In [ ]:
import json, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentinelhub import SentinelHubRequest, DataCollection, MimeType, CRS, BBox
from pyproj import Transformer

# ===== KONFIGURASI CITRA (Copernicus Data Space / Sentinel Hub) =====
TGL_MULAI, TGL_AKHIR = "2025-06-01", "2025-09-30"   # musim kemarau; makin panjang = makin banyak scene = makin boros PU
MAX_CLOUD   = 30        # persen maksimum tutupan awan per scene
N_THREAD    = 4         # request paralel (batas gratis ~300 request/menit)
N_SCENE_EST = 20        # perkiraan jumlah scene per piksel (hanya untuk estimasi biaya)
GRID_CHIP_M = 1500      # titik dalam satu poligon dan satu sel grid ini digabung jadi 1 request (makin besar = request makin sedikit, PU makin besar)
# =====================================================================

# Komposit median: awan/bayangan/cirrus dibuang lewat SCL (3, 8, 9, 10). Output = reflektansi x 10000 (UINT16).
EVALSCRIPT = """//VERSION=3
// Komposit median Sentinel-2 L2A: masking awan/bayangan/cirrus via SCL, output reflektansi x 10000 (UINT16)
function setup() {
  return {
    input: [{ bands: ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12","SCL","dataMask"] }],
    output: { bands: 10, sampleType: "UINT16" },
    mosaicking: "ORBIT"
  };
}
function median(a) {
  a.sort(function (x, y) { return x - y; });
  var n = a.length;
  return (n % 2) ? a[(n - 1) / 2] : (a[n / 2 - 1] + a[n / 2]) / 2;
}
function evaluatePixel(samples) {
  var names = ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"];
  var vals = [[],[],[],[],[],[],[],[],[],[]];
  for (var i = 0; i < samples.length; i++) {
    var s = samples[i];
    if (s.dataMask === 1 && s.SCL !== 3 && s.SCL !== 8 && s.SCL !== 9 && s.SCL !== 10) {
      for (var b = 0; b < 10; b++) { vals[b].push(s[names[b]]); }
    }
  }
  var out = [];
  for (var b = 0; b < 10; b++) {
    out.push(vals[b].length === 0 ? 0 : Math.round(median(vals[b]) * 10000));
  }
  return out;
}
"""

BAND_SH = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]   # = BAND_ASLI (B2..B12)

def ambil_array(x0, y0, x1, y1, w, h, crs):
    """Unduh komposit median Sentinel-2 untuk bbox (x0,y0,x1,y1) berukuran w x h piksel -> array (h, w, 10) uint16."""
    s2 = DataCollection.SENTINEL2_L2A.define_from("s2l2a_cdse", service_url=config.sh_base_url)
    req = SentinelHubRequest(
        evalscript=EVALSCRIPT,
        input_data=[SentinelHubRequest.input_data(data_collection=s2,
                                                  time_interval=(TGL_MULAI, TGL_AKHIR),
                                                  maxcc=MAX_CLOUD / 100)],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=BBox((x0, y0, x1, y1), crs=crs),
        size=(w, h),
        config=config,
    )
    arr = np.asarray(req.get_data(max_threads=1)[0])
    return arr[..., None] if arr.ndim == 2 else arr

def ambil_dengan_ulang(fn, *args, tries=3):
    for k in range(tries):
        try:
            return fn(*args)
        except Exception as e:
            if k == tries - 1:
                raise
            print("   ulang", k + 1, "->", str(e)[:140]); time.sleep(5 * (k + 1))

CRS_UTM49S = CRS(32749)
to_utm49 = Transformer.from_crs(4326, 32749, always_xy=True)

def buat_chip(df_pts, grid_m=None):
    """Kelompokkan titik per (poligon, sel grid 1,5 km). Satu kelompok = satu request potongan citra kecil
    (bbox dibungkus ketat di sekitar titik, selaras grid 10 m Sentinel-2)."""
    grid_m = grid_m or GRID_CHIP_M
    d = df_pts.copy()
    d["x"], d["y"] = to_utm49.transform(d.lon.values, d.lat.values)
    d["cx"] = np.floor(d.x / grid_m).astype(int)
    d["cy"] = np.floor(d.y / grid_m).astype(int)
    chips = []
    for _, g in d.groupby(["poly_id", "cx", "cy"]):
        x0 = np.floor(g.x.min() / 10) * 10;  x1 = np.floor(g.x.max() / 10) * 10 + 10
        y0 = np.floor(g.y.min() / 10) * 10;  y1 = np.floor(g.y.max() / 10) * 10 + 10
        chips.append(dict(idx=g.index.values, xs=g.x.values, ys=g.y.values,
                          x0=x0, y0=y0, x1=x1, y1=y1, w=int((x1 - x0) / 10), h=int((y1 - y0) / 10)))
    return chips

def estimasi_pu(chips):
    # 1 PU = 512x512 px, 3 band input, 16-bit, 1 scene; minimal 0,005 PU per request
    return sum(max(0.005, c["w"] * c["h"] / 262144 * (12 / 3) * N_SCENE_EST) for c in chips)

def proses_chip(c):
    arr = ambil_dengan_ulang(ambil_array, c["x0"], c["y0"], c["x1"], c["y1"], c["w"], c["h"], CRS_UTM49S)
    cols = np.clip(((c["xs"] - c["x0"]) // 10).astype(int), 0, c["w"] - 1)
    rows = np.clip(((c["y1"] - c["ys"]) // 10).astype(int), 0, c["h"] - 1)
    vals = arr[rows, cols, :].astype("float64")
    vals[(vals == 0).all(axis=1)] = np.nan              # piksel tanpa data (awan terus-menerus)
    return pd.DataFrame(vals, columns=BAND_ASLI, index=c["idx"])

def ekstrak_titik(df_pts, chips, n_thread=None, label=""):
    hasil, gagal, t0 = [], [], time.time()
    with ThreadPoolExecutor(max_workers=n_thread or N_THREAD) as ex:
        fut = {ex.submit(proses_chip, c): i for i, c in enumerate(chips)}
        for n, f in enumerate(as_completed(fut), 1):
            try:
                hasil.append(f.result())
            except Exception as e:
                gagal.append((fut[f], str(e)[:120]))
            if n % 25 == 0 or n == len(chips):
                print(f"{label}Chip {n}/{len(chips)} | gagal {len(gagal)} | {time.time()-t0:.0f} dtk", end="\r")
    print()
    if gagal:
        print(f"[!] {len(gagal)} chip gagal, contoh:", gagal[0])
    return (pd.concat(hasil).sort_index() if hasil else pd.DataFrame(columns=BAND_ASLI)), gagal

In [ ]:
# Uji koneksi: ambil nilai 1 piksel (sekitar Surabaya). Harus keluar 10 angka reflektansi x 10000.
x, y = to_utm49.transform(112.7508, -7.2575)
x0, y0 = np.floor(x / 10) * 10, np.floor(y / 10) * 10
uji = ambil_array(x0, y0, x0 + 10, y0 + 10, 1, 1, CRS_UTM49S)
print("Bentuk array:", uji.shape, "| dtype:", uji.dtype)
print(dict(zip(BAND_ASLI, uji.reshape(-1, uji.shape[-1])[0].tolist())))
print("Koneksi ke Copernicus berhasil.")

### Estimasi biaya dan uji kecil
Jalankan estimasi dulu. Lalu uji 20 chip untuk memastikan semuanya benar, dan cek pemakaian PU di dashboard sebelum menarik seluruh data.

In [ ]:
chips = buat_chip(pts)
pu = estimasi_pu(chips)
luas_chip = np.array([c["w"] * c["h"] for c in chips])
print(f"Titik: {len(pts)} | request (chip): {len(chips)} | piksel/chip rata-rata: {luas_chip.mean():.0f}, maks: {luas_chip.max()}")
print(f"Estimasi biaya: ~{pu:,.0f} PU (asumsi {N_SCENE_EST} scene per piksel; kuota gratis ~10.000 PU/bulan)")

PU_BUDGET_TITIK = 4000
assert pu <= PU_BUDGET_TITIK, "Estimasi PU terlalu besar: kecilkan N_PER_CLASS / MAX_PER_POLY atau persempit TGL_MULAI-TGL_AKHIR."

In [ ]:
# Uji kecil: 20 chip pertama
uji_df, uji_gagal = ekstrak_titik(pts, chips[:20], label="[uji] ")
print("Titik terisi:", uji_df.dropna().shape[0], "dari", uji_df.shape[0])
display(uji_df.dropna().describe().T[["mean", "min", "max"]].round(0))
print(f"Estimasi 20 chip ini: ~{estimasi_pu(chips[:20]):.1f} PU -> bandingkan dengan pemakaian di dashboard.")

### Ekstraksi penuh dan pembentukan dataset

In [ ]:
df_val, gagal = ekstrak_titik(pts, chips)

if gagal:                                           # ulangi chip yang gagal satu kali
    print("Mengulang chip yang gagal...")
    ulang, gagal2 = ekstrak_titik(pts, [chips[i] for i, _ in gagal], n_thread=2, label="[ulang] ")
    df_val = pd.concat([df_val, ulang]).sort_index()

df_raw = pts.join(df_val).dropna(subset=BAND_ASLI).reset_index(drop=True)
df_raw[BAND_ASLI] = df_raw[BAND_ASLI] / 10000.0     # reflektansi x 10000 -> 0-1

df_dataset = hitung_indeks_spektral(df_raw)
df_dataset = df_dataset[["poly_id", "kelas_id", "kelas", "lon", "lat"] + FITUR_SEMUA]
df_dataset.to_csv("dataset_pixel_sentinel2_jatim.csv", index=False)

print(f"\nTitik dikirim: {len(pts)} | titik valid: {len(df_dataset)}")
print("Tersimpan: dataset_pixel_sentinel2_jatim.csv")
display(df_dataset.groupby("kelas").size().reindex(NAMA_KELAS).to_frame("Jumlah piksel valid"))
display(df_dataset.head())

---
## Validasi dan Visualisasi Dataset
Pengecekan NaN/Inf, rekap per kelas, statistik deskriptif, sebaran spasial, tanda tangan spektral (rata-rata nyata dari piksel), dan separabilitas indeks. Tabel rata-rata indeks per kelas harus masuk akal: air punya NDWI/MNDWI tinggi, vegetasi punya NDVI tinggi, bangunan punya NDBI tertinggi.

In [ ]:
nan_total = df_dataset[FITUR_SEMUA].isnull().sum().sum()
inf_total = np.isinf(df_dataset[FITUR_SEMUA].to_numpy()).sum()
print("Pengecekan Kualitas Data:")
print(f"  - Nilai Kosong (NaN) : {nan_total} (Valid: {nan_total == 0})")
print(f"  - Nilai Tak Hingga   : {inf_total} (Valid: {inf_total == 0})")

rekap_kelas = (df_dataset.groupby(["kelas_id", "kelas"])
               .agg(Jumlah_Piksel=("poly_id", "size"), Jumlah_Poligon=("poly_id", "nunique")))
rekap_kelas["Persentase (%)"] = (rekap_kelas["Jumlah_Piksel"] / len(df_dataset) * 100).round(2)
display(rekap_kelas)

print("\nStatistik Deskriptif 10 Band dan 8 Indeks Spektral:")
display(df_dataset[FITUR_SEMUA].describe().T[["mean", "std", "min", "50%", "max"]].round(4))

In [ ]:
panjang_gelombang = [492, 560, 665, 704, 740, 783, 833, 865, 1614, 2202]

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))

# 1) sebaran spasial titik sampel
for nama in NAMA_KELAS:
    sub = df_dataset[df_dataset["kelas"] == nama]
    axes[0].scatter(sub["lon"], sub["lat"], s=6, alpha=.6, color=WARNA_KELAS[nama], label=nama)
axes[0].set_title("Sebaran Spasial Sampel Piksel", fontweight="bold")
axes[0].set_xlabel("Bujur (BT)"); axes[0].set_ylabel("Lintang (LS)")
axes[0].legend(markerscale=3, fontsize=8)

# 2) tanda tangan spektral (rata-rata nyata dari piksel)
for nama in NAMA_KELAS:
    m = df_dataset[df_dataset["kelas"] == nama][BAND_ASLI].mean().values
    axes[1].plot(panjang_gelombang, m, marker="o", lw=2.2, color=WARNA_KELAS[nama], label=nama)
axes[1].set_title("Tanda Tangan Spektral (rata-rata piksel)", fontweight="bold")
axes[1].set_xlabel("Panjang gelombang (nm)"); axes[1].set_ylabel("Reflektansi (0-1)")
axes[1].grid(True, ls="--", alpha=.5); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, idx in zip(axes.flatten(), ["NDVI", "NDWI", "MNDWI", "NDBI"]):
    sns.boxplot(data=df_dataset, x="kelas", y=idx, order=NAMA_KELAS, palette=WARNA_KELAS, ax=ax)
    ax.set_title(f"Distribusi {idx}", fontweight="bold"); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
plt.suptitle("Separabilitas Antar Kelas Berdasarkan Indeks Spektral", fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

---
## Pembagian Data 70 / 15 / 15 dan Penskalaan Fitur
Pembagian dilakukan **per poligon** (bukan per piksel) di tiap kelas, agar piksel bertetangga dari satu poligon tidak muncul di training dan test sekaligus (*spatial leakage*). Proporsi piksel aktual bisa meleset sedikit dari 70/15/15. `StandardScaler` hanya di-fit pada data latih dan dipakai untuk KNN.

In [ ]:
# Split 70/15/15 PER POLIGON (bukan per piksel) agar piksel bertetangga tidak bocor antar split
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
rng_split = np.random.default_rng(SEED)

split_map = {}
for kid, g in df_dataset.groupby("kelas_id"):
    ids = g.poly_id.unique().copy()
    rng_split.shuffle(ids)
    n = len(ids)
    if n < 3:
        print(f"[!] Kelas {g.kelas.iloc[0]} hanya {n} poligon -> semua masuk train")
        n_val = n_test = 0
    else:
        n_val  = max(1, int(round(n * VAL_FRAC)))
        n_test = max(1, int(round(n * TEST_FRAC)))
    for pid in ids[:n_test]:               split_map[pid] = "test"
    for pid in ids[n_test:n_test + n_val]: split_map[pid] = "val"
    for pid in ids[n_test + n_val:]:       split_map[pid] = "train"

df_dataset["split"] = df_dataset.poly_id.map(split_map)

tr = df_dataset[df_dataset.split == "train"]
va = df_dataset[df_dataset.split == "val"]
te = df_dataset[df_dataset.split == "test"]

X_train, y_train = tr[FITUR_SEMUA], tr["kelas"]
X_val,   y_val   = va[FITUR_SEMUA], va["kelas"]
X_test,  y_test  = te[FITUR_SEMUA], te["kelas"]

print("=" * 60)
print(f"HASIL PEMBAGIAN DATA (Total: {len(df_dataset)} piksel dari {df_dataset.poly_id.nunique()} poligon)")
print("=" * 60)
for nm, d in [("Training", tr), ("Validation", va), ("Testing", te)]:
    print(f"{nm:<11}: {len(d):>6} piksel ({len(d)/len(df_dataset)*100:4.1f}%) | {d.poly_id.nunique()} poligon")

rekap_split = pd.DataFrame({
    "Training":   y_train.value_counts(),
    "Validation": y_val.value_counts(),
    "Testing":    y_test.value_counts(),
}).reindex(NAMA_KELAS).fillna(0).astype(int)
rekap_split["Total"] = rekap_split.sum(axis=1)
display(rekap_split)
print("Poligon per kelas dan split:")
display(df_dataset.groupby(["kelas", "split"]).poly_id.nunique().unstack(fill_value=0)
        .reindex(NAMA_KELAS)[["train", "val", "test"]])

df_dataset.to_csv("dataset_pixel_sentinel2_jatim.csv", index=False)   # kolom 'split' ikut tersimpan

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, cohen_kappa_score, f1_score)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)     # fit hanya di data latih
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
print("Penskalaan selesai. Dimensi X_train_scaled:", X_train_scaled.shape)

---
# MODEL 1: Gaussian Naive Bayes
Memodelkan distribusi tiap fitur per kelas dengan kurva normal dan asumsi fitur saling bebas. Hyperparameter `var_smoothing` di-tuning pada Validation Set. Tidak memerlukan penskalaan.

In [ ]:
# ===== MODEL 1: GAUSSIAN NAIVE BAYES =====
variasi_smoothing = np.logspace(-12, -1, 12)
skor_nb_val = []
for vs in variasi_smoothing:
    m = GaussianNB(var_smoothing=vs).fit(X_train, y_train)
    skor_nb_val.append(f1_score(y_val, m.predict(X_val), average="weighted"))

best_var_smoothing = variasi_smoothing[int(np.argmax(skor_nb_val))]
print(f"var_smoothing terbaik : {best_var_smoothing:.2e}")
print(f"F1 validasi tertinggi : {max(skor_nb_val) * 100:.2f}%")

plt.figure(figsize=(8, 4))
plt.semilogx(variasi_smoothing, np.array(skor_nb_val) * 100, marker="o", color="#E74C3C")
plt.title("Tuning var_smoothing Naive Bayes (Validation Set)", fontweight="bold")
plt.xlabel("var_smoothing"); plt.ylabel("F1 weighted (%)"); plt.grid(alpha=.4)
plt.tight_layout(); plt.show()

model_gnb = GaussianNB(var_smoothing=best_var_smoothing).fit(X_train, y_train)
pred_gnb_val = model_gnb.predict(X_val)
acc_gnb_val = accuracy_score(y_val, pred_gnb_val)
f1_gnb_val  = f1_score(y_val, pred_gnb_val, average="weighted")
print("\n[Evaluasi Validation Set - Gaussian Naive Bayes]")
print(f"  - Akurasi  : {acc_gnb_val * 100:.2f}%")
print(f"  - F1-Score : {f1_gnb_val * 100:.2f}%")

---
# MODEL 2: K-Nearest Neighbors (KNN)
Klasifikasi berdasarkan suara mayoritas k tetangga terdekat (jarak Euclidean) pada fitur yang sudah diskalakan. Nilai k di-tuning pada Validation Set.

In [ ]:
# ===== MODEL 2: K-NEAREST NEIGHBORS =====
daftar_k = [1, 3, 5, 7, 9, 11, 15, 21, 31]
skor_tr, skor_va = [], []
for k in daftar_k:
    m = KNeighborsClassifier(n_neighbors=k, metric="euclidean", n_jobs=-1).fit(X_train_scaled, y_train)
    skor_tr.append(accuracy_score(y_train, m.predict(X_train_scaled)))
    skor_va.append(f1_score(y_val, m.predict(X_val_scaled), average="weighted"))

plt.figure(figsize=(9, 4.5))
plt.plot(daftar_k, np.array(skor_tr) * 100, marker="o", label="Akurasi Training", color="#3498DB")
plt.plot(daftar_k, np.array(skor_va) * 100, marker="s", label="F1 Validation", color="#E74C3C")
plt.title("Tuning k pada KNN", fontweight="bold")
plt.xlabel("k"); plt.ylabel("Skor (%)"); plt.xticks(daftar_k); plt.grid(alpha=.4); plt.legend()
plt.tight_layout(); plt.show()

best_k = daftar_k[int(np.argmax(skor_va))]
print("k terbaik:", best_k)

model_knn = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", n_jobs=-1).fit(X_train_scaled, y_train)
pred_knn_val = model_knn.predict(X_val_scaled)
acc_knn_val = accuracy_score(y_val, pred_knn_val)
f1_knn_val  = f1_score(y_val, pred_knn_val, average="weighted")
print(f"\n[Evaluasi Validation Set - KNN (k={best_k})]")
print(f"  - Akurasi  : {acc_knn_val * 100:.2f}%")
print(f"  - F1-Score : {f1_knn_val * 100:.2f}%")

---
## Komparasi Model dan Evaluasi Akhir pada Testing Set
Data uji tidak dipakai saat pelatihan maupun tuning. Metrik: akurasi, F1, Cohen's Kappa, classification report, dan confusion matrix.

In [ ]:
# --- Komparasi di data validasi ---
display(pd.DataFrame({
    "Model": ["Gaussian Naive Bayes", f"KNN (k={best_k})"],
    "Hyperparameter": [f"var_smoothing={best_var_smoothing:.2e}", f"n_neighbors={best_k}, euclidean"],
    "Akurasi Validasi (%)": [round(acc_gnb_val * 100, 2), round(acc_knn_val * 100, 2)],
    "F1 Validasi (%)": [round(f1_gnb_val * 100, 2), round(f1_knn_val * 100, 2)],
}).set_index("Model"))

# --- Evaluasi akhir di data test ---
pred_gnb_test = model_gnb.predict(X_test)
pred_knn_test = model_knn.predict(X_test_scaled)

acc_gnb_test = accuracy_score(y_test, pred_gnb_test)
f1_gnb_test  = f1_score(y_test, pred_gnb_test, average="weighted")
kappa_gnb    = cohen_kappa_score(y_test, pred_gnb_test)
acc_knn_test = accuracy_score(y_test, pred_knn_test)
f1_knn_test  = f1_score(y_test, pred_knn_test, average="weighted")
kappa_knn    = cohen_kappa_score(y_test, pred_knn_test)

tabel_test = pd.DataFrame({
    "Model": ["Gaussian Naive Bayes", f"KNN (k={best_k})"],
    "Akurasi Test (%)": [round(acc_gnb_test * 100, 2), round(acc_knn_test * 100, 2)],
    "F1 Test (%)": [round(f1_gnb_test * 100, 2), round(f1_knn_test * 100, 2)],
    "Cohen's Kappa": [round(kappa_gnb, 4), round(kappa_knn, 4)],
}).set_index("Model")
print("=" * 60); print(f"EVALUASI AKHIR PADA TESTING SET ({len(y_test)} piksel, {te.poly_id.nunique()} poligon)"); print("=" * 60)
display(tabel_test)

print("\n--- CLASSIFICATION REPORT: GAUSSIAN NAIVE BAYES ---")
print(classification_report(y_test, pred_gnb_test, labels=NAMA_KELAS, target_names=NAMA_KELAS, digits=4, zero_division=0))
print("--- CLASSIFICATION REPORT: KNN ---")
print(classification_report(y_test, pred_knn_test, labels=NAMA_KELAS, target_names=NAMA_KELAS, digits=4, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, pr, nm, cmap, acc in [(axes[0], pred_gnb_test, "Gaussian Naive Bayes", "Blues", acc_gnb_test),
                              (axes[1], pred_knn_test, f"KNN (k={best_k})", "Greens", acc_knn_test)]:
    cm = confusion_matrix(y_test, pr, labels=NAMA_KELAS)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, cbar=False, ax=ax,
                xticklabels=NAMA_KELAS, yticklabels=NAMA_KELAS, annot_kws={"size": 11, "weight": "bold"})
    ax.set_title(f"Confusion Matrix: {nm}\n(Akurasi Test: {acc*100:.2f}%)", fontweight="bold")
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

---
## Penyiapan Citra Seluruh Area Studi (untuk Peta)
Komposit Sentinel-2 seluruh AOI diminta dari Copernicus per tile langsung ke memori (tanpa menyimpan TIF), pada resolusi `RES_M`. Dipakai bersama oleh kedua model. Sel ini menampilkan estimasi PU lebih dulu dan berhenti bila melebihi `PU_BUDGET`.

In [ ]:
# ===== KONFIGURASI PETA =====
RES_M   = 200      # meter/piksel peta. 200 = hemat | 100 = lebih detail tetapi ~4x lebih boros PU
TILE_PX = 700      # ukuran tile permintaan (piksel); maksimum Sentinel Hub 2500
PU_BUDGET = 3000   # batas aman estimasi PU untuk peta (kuota gratis total ~10.000 PU/bulan)
# ============================

t_fw = Transformer.from_crs(4326, 3857, always_xy=True)
t_bw = Transformer.from_crs(3857, 4326, always_xy=True)
xmin, ymin = t_fw.transform(AOI["west"], AOI["south"])
xmax, ymax = t_fw.transform(AOI["east"], AOI["north"])

X0 = np.floor(xmin / RES_M) * RES_M
Y1 = np.ceil(ymax / RES_M) * RES_M
W = int(np.ceil((xmax - X0) / RES_M))
H = int(np.ceil((Y1 - ymin) / RES_M))

grid = [(c, r) for r in range(0, H, TILE_PX) for c in range(0, W, TILE_PX)]
pu_tile = [min(TILE_PX, W - c) * min(TILE_PX, H - r) / 262144 * (12 / 3) * N_SCENE_EST for c, r in grid]
print(f"Grid peta: {W} x {H} = {W*H/1e6:.2f} juta piksel | {len(grid)} tile | estimasi ~{sum(pu_tile):,.0f} PU "
      f"(asumsi {N_SCENE_EST} scene/piksel)")
assert sum(pu_tile) <= PU_BUDGET, "Estimasi PU melebihi PU_BUDGET. Naikkan RES_M, persempit tanggal, atau naikkan PU_BUDGET."

mosaic = np.full((len(BAND_ASLI), H, W), np.nan, dtype="float32")

def unduh_tile_peta(c0, r0):
    c1, r1 = min(c0 + TILE_PX, W), min(r0 + TILE_PX, H)
    bx0, bx1 = X0 + c0 * RES_M, X0 + c1 * RES_M
    by1, by0 = Y1 - r0 * RES_M, Y1 - r1 * RES_M
    arr = ambil_dengan_ulang(ambil_array, bx0, by0, bx1, by1, c1 - c0, r1 - r0, CRS.POP_WEB)
    return c0, r0, arr

gagal = []
t0 = time.time()
with ThreadPoolExecutor(max_workers=min(N_THREAD, 3)) as ex:
    fut = {ex.submit(unduh_tile_peta, c, r): (c, r) for c, r in grid}
    for n, f in enumerate(as_completed(fut), 1):
        try:
            c0, r0, arr = f.result()
            h, w, _ = arr.shape
            mosaic[:, r0:r0 + h, c0:c0 + w] = np.transpose(arr, (2, 0, 1)).astype("float32")
        except Exception as e:
            gagal.append((fut[f], str(e)[:120]))
        print(f"Tile {n}/{len(grid)} | gagal {len(gagal)} | {time.time()-t0:.0f} dtk", end="\r")

mosaic[:, (mosaic == 0).all(axis=0)] = np.nan             # piksel tanpa data
valid = np.isfinite(mosaic).all(axis=0)
print(f"\nTile gagal: {len(gagal)} | piksel valid: {valid.sum():,} dari {H*W:,} ({valid.mean()*100:.1f}%)")
if gagal:
    print("[!] Jalankan ulang sel ini untuk mengulang. Contoh error:", gagal[0])

---
## Fungsi Prediksi Raster dan Pembuat Peta Folium (dipakai kedua model)

In [ ]:
import folium
from folium.raster_layers import ImageOverlay

CH = 200_000     # jumlah piksel per potongan prediksi (hemat memori)

def fitur_mosaic(raw):
    # raw: nilai mentah (reflektansi x 10000) -> reflektansi 0-1 -> 18 fitur (urutan = FITUR_SEMUA)
    d = pd.DataFrame(raw / 10000.0, columns=BAND_ASLI)
    return hitung_indeks_spektral(d)[FITUR_SEMUA]

def prediksi_mosaic(fn_predict):
    # fn_predict: fungsi (DataFrame fitur) -> array nama kelas. Hasil: raster (H, W), 0 = tidak ada data
    flat = mosaic.reshape(len(BAND_ASLI), -1).T
    vi = np.flatnonzero(valid.ravel())
    out = np.zeros(H * W, dtype="uint8")
    t0 = time.time()
    for s in range(0, len(vi), CH):
        sel = vi[s:s + CH]
        pred = fn_predict(fitur_mosaic(flat[sel].astype("float64")))
        out[sel] = pd.Series(pred).map(NAMA_KE_ID).to_numpy(dtype="uint8")
        print(f"Prediksi {min(s + CH, len(vi)):,}/{len(vi):,} piksel | {time.time()-t0:.0f} dtk", end="\r")
    print()
    return out.reshape(H, W)

def hex_rgb(h):
    h = h.lstrip("#"); return [int(h[i:i + 2], 16) for i in (0, 2, 4)]

def buat_peta(klas, judul, nama_file, teks_info, gdf_poly=None, kol_pred=None):
    """klas: raster kelas (H, W). gdf_poly: (opsional) poligon; bila kol_pred diisi, poligon diwarnai menurut prediksi."""
    rgba = np.zeros((H, W, 4), dtype=np.uint8)
    for kid, v in KELAS.items():
        rgba[klas == kid] = hex_rgb(v["warna"]) + [255]

    sat = np.zeros((H, W, 4), dtype=np.uint8)               # RGB asli dari B4, B3, B2
    for k, b in enumerate(["B4", "B3", "B2"]):
        band = np.where(valid, mosaic[BAND_ASLI.index(b)], 0)
        sat[..., k] = np.clip((band / 3000) ** 0.8 * 255, 0, 255).astype(np.uint8)
    sat[..., 3] = np.where(valid, 255, 0)

    west, north = t_bw.transform(X0, Y1)
    east, south = t_bw.transform(X0 + W * RES_M, Y1 - H * RES_M)
    bounds = [[south, west], [north, east]]

    m = folium.Map(location=[(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2],
                   zoom_start=8, tiles=None, control_scale=True)
    m.get_root().header.add_child(folium.Element("<style>.lulc-pix{image-rendering:pixelated}</style>"))

    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Tiles © Esri — Source: Esri, Maxar, Earthstar Geographics",
                     name="Esri Satellite (online)", overlay=False, max_zoom=19).add_to(m)
    folium.TileLayer("https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}", attr="Google",
                     name="Google Hybrid (dengan label)", overlay=False, show=False, max_zoom=20).add_to(m)

    ImageOverlay(image=sat, bounds=bounds, opacity=1.0, name="Satelit Sentinel-2 (tertanam)",
                 show=False, interactive=False, class_name="lulc-pix").add_to(m)
    ImageOverlay(image=rgba, bounds=bounds, opacity=0.8, name=f"Hasil Klasifikasi: {judul}",
                 interactive=False, class_name="lulc-pix").add_to(m)

    if gdf_poly is not None:                                # poligon training / hasil prediksi per poligon
        gp = gdf_poly.copy()
        gp["geometry"] = gp.geometry.simplify(0.002)
        if kol_pred:
            kolom = ["kelas", kol_pred, "geometry"]
            warna_fn = lambda f: {"fillColor": WARNA_KELAS.get(f["properties"][kol_pred], "#888"),
                                  "color": "white", "weight": 1, "fillOpacity": .65}
            tip = folium.GeoJsonTooltip(fields=["kelas", kol_pred], aliases=["Aktual", "Prediksi"])
            nama_lyr = f"Poligon (warna = prediksi {judul})"
        else:
            kolom = ["kelas", "poly_id", "geometry"]
            warna_fn = lambda f: {"fillColor": WARNA_KELAS.get(f["properties"]["kelas"], "#888"),
                                  "color": "black", "weight": 1, "fillOpacity": .35}
            tip = folium.GeoJsonTooltip(fields=["kelas", "poly_id"], aliases=["Kelas", "ID poligon"])
            nama_lyr = "Poligon training (warna = kelas)"
        folium.GeoJson(gp[kolom].to_json(), style_function=warna_fn, tooltip=tip,
                       name=nama_lyr, show=False).add_to(m)

    for kid, v in KELAS.items():                             # titik sampel per kelas
        fg = folium.FeatureGroup(name=f"Sampel: {v['nama']}", show=False)
        d = df_dataset[df_dataset.kelas_id == kid]
        d = d.sample(min(300, len(d)), random_state=1)
        for _, p in d.iterrows():
            folium.CircleMarker([p.lat, p.lon], radius=3, color="white", weight=1, fill=True,
                                fill_color=v["warna"], fill_opacity=.95,
                                tooltip=f"{v['nama']}").add_to(fg)
        fg.add_to(m)

    fg_aoi = folium.FeatureGroup(name="Batas area studi")
    folium.Rectangle([[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]], color="white", weight=2,
                     dash_array="8 6", fill=False).add_to(fg_aoi)
    fg_aoi.add_to(m)

    items = "".join(
        f'<div style="margin:3px 0"><span style="display:inline-block;width:16px;height:16px;'
        f'background:{v["warna"]};border:1px solid #666;vertical-align:middle;margin-right:8px"></span>{v["nama"]}</div>'
        for v in KELAS.values())
    legend = ('<div style="position:fixed;bottom:40px;left:20px;z-index:9999;background:white;padding:10px 14px;'
              'border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.4);font:13px Arial"><b>Legenda LULC</b>'
              + items + "</div>")
    info = ('<div style="position:fixed;top:12px;left:60px;z-index:9999;background:white;padding:6px 12px;'
            'border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.4);font:12px Arial">' + teks_info + "</div>")
    m.get_root().html.add_child(folium.Element(legend))
    m.get_root().html.add_child(folium.Element(info))
    folium.LayerControl(collapsed=False).add_to(m)

    m.save(nama_file)
    print(f"Peta disimpan: {nama_file} ({os.path.getsize(nama_file)/1e6:.1f} MB) - bisa ditanam di web statis lewat <iframe>")
    return m

---
## Peta Hasil Klasifikasi: Gaussian Naive Bayes
Model Naive Bayes memprediksi seluruh piksel area studi, ditampilkan di atas basemap satelit Esri / Google Hybrid. Layer dapat dinyalakan/dimatikan lewat kontrol di kanan atas (hasil klasifikasi, satelit Sentinel-2 tertanam, poligon training, titik sampel per kelas, batas area studi).

In [ ]:
klas_gnb = prediksi_mosaic(lambda X: model_gnb.predict(X))
print("Selesai - Naive Bayes")
print(pd.Series(klas_gnb[klas_gnb > 0]).map({k: v["nama"] for k, v in KELAS.items()}).value_counts())

In [ ]:
metrik_gnb = {"Akurasi": acc_gnb_test, "F1": f1_gnb_test, "Kappa": kappa_gnb}
info_gnb = (f"<b>Model:</b> Gaussian Naive Bayes | <b>Akurasi test:</b> {acc_gnb_test*100:.1f}% | "
            f"<b>F1:</b> {f1_gnb_test:.3f} | <b>Kappa:</b> {kappa_gnb:.3f} | <b>Resolusi:</b> {RES_M} m")
peta_gnb = buat_peta(klas_gnb, "Gaussian Naive Bayes", "peta_pixel_gnb.html", info_gnb, gdf_poly=gdf)
peta_gnb

---
## Peta Hasil Klasifikasi: K-Nearest Neighbors
Model KNN (fitur diskalakan dengan `scaler` yang sama dengan saat pelatihan) memprediksi seluruh piksel area studi.

In [ ]:
klas_knn = prediksi_mosaic(lambda X: model_knn.predict(scaler.transform(X)))
print("Selesai - KNN")
print(pd.Series(klas_knn[klas_knn > 0]).map({k: v["nama"] for k, v in KELAS.items()}).value_counts())

In [ ]:
metrik_knn = {"Akurasi": acc_knn_test, "F1": f1_knn_test, "Kappa": kappa_knn}
info_knn = (f"<b>Model:</b> KNN (k={best_k}) | <b>Akurasi test:</b> {acc_knn_test*100:.1f}% | "
            f"<b>F1:</b> {f1_knn_test:.3f} | <b>Kappa:</b> {kappa_knn:.3f} | <b>Resolusi:</b> {RES_M} m")
peta_knn = buat_peta(klas_knn, f"KNN (k={best_k})", "peta_pixel_knn.html", info_knn, gdf_poly=gdf)
peta_knn

---
## Luas Tiap Kelas per Model
Luas dihitung dari jumlah piksel peta dikali luas piksel (dikoreksi lintang). Berguna untuk analisis kebijakan, misalnya luas sawah dan mangrove menurut tiap model.

In [ ]:
lat_row = t_bw.transform(np.full(H, X0), Y1 - (np.arange(H) + 0.5) * RES_M)[1]
km2_px  = (RES_M * np.cos(np.radians(lat_row))) ** 2 / 1e6        # luas piksel per baris (km2)

def luas_kelas(klas):
    return {v["nama"]: float(((klas == k).sum(axis=1) * km2_px).sum()) for k, v in KELAS.items()}

df_luas = pd.DataFrame({"Naive Bayes (km2)": luas_kelas(klas_gnb),
                        f"KNN k={best_k} (km2)": luas_kelas(klas_knn)}).reindex(NAMA_KELAS)
for c in list(df_luas.columns):
    df_luas[c.replace("(km2)", "(%)")] = (df_luas[c] / df_luas[c].sum() * 100).round(2)
df_luas = df_luas.round(1)
display(df_luas)
df_luas.to_csv("luas_per_kelas_per_model.csv")
print("Tersimpan: luas_per_kelas_per_model.csv")

---
## Kesimpulan
*Tulis setelah seluruh sel dijalankan, berdasarkan hasil nyata:*
1. **Dataset:** jumlah piksel dan poligon per kelas, serta pembagian training/validasi/test.
2. **Performa model:** bandingkan akurasi, F1, dan Kappa Naive Bayes vs KNN pada data test; sebutkan model terbaik.
3. **Kesalahan klasifikasi:** kelas yang sering tertukar menurut confusion matrix dan kemungkinan penyebabnya.
4. **Keterbatasan:** resolusi peta (`RES_M`) lebih kasar dari resolusi data latih (10 m); periode citra hanya musim kemarau; jumlah poligon per kelas.
5. **Implikasi kebijakan:** gunakan tabel luas per kelas.